# P3a and behavioral statistics

This notebook analyzes the target-locked EEG response and saccadic behavior. It prepares Tables 1–5, the participant-level specificity analyses, and Figures 2–3.

## Inputs

- `saccades_summary.xlsx` from `01_saccades.ipynb`.
- `eeg_features.xlsx` and the `evoked` folder from `02_eeg_features.ipynb`.
- A participant-level Excel file containing age, diagnostic group, four SIFS domain scores, and five FFiCD domain scores. Include a row for every EEG participant, leaving unavailable questionnaire scores blank.

Set the paths and column mapping below. Then set `RUN_ANALYSES = True`, restart the kernel, and run all cells. Inspect model convergence, sample sizes, score coverage, and the result tables before exporting. To save results, set `EXPORT_OUTPUTS = True` and run the export cell.

With `RUN_ANALYSES = False`, no participant data are loaded and no statistical models are fitted. Python dependencies are `numpy`, `pandas`, `scipy`, `statsmodels`, `matplotlib`, `mne`, `openpyxl`, and Jupyter.

In [1]:
from pathlib import Path
from importlib.metadata import version
import hashlib
import platform
import warnings

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import mne
from IPython.display import Markdown, display

software_versions = pd.DataFrame(
    [("Python", platform.python_version())] + [(name, version(name)) for name in
     ("numpy", "pandas", "scipy", "statsmodels", "matplotlib", "mne", "openpyxl")],
    columns=["software", "version"],
)
pd.set_option("display.max_rows", 25)
pd.set_option("display.max_columns", 16)
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "pdf.fonttype": 42, "ps.fonttype": 42})
display(software_versions)

,software,version
0,Python,3.12.14
1,numpy,2.3.5
2,pandas,2.2.3
3,scipy,1.17.0
4,statsmodels,0.15.0
5,matplotlib,3.10.8
6,mne,1.13.2
7,openpyxl,3.1.5


In [2]:
RUN_ANALYSES = False
BEHAVIOR_SUMMARY_FILE = Path("PATH_TO_SACCADE_OUTPUTS/saccades_summary.xlsx").expanduser()
EEG_FEATURES_FILE = Path("PATH_TO_EEG_OUTPUTS/eeg_features.xlsx").expanduser()
EVOKED_DIR = Path("PATH_TO_EEG_OUTPUTS/evoked").expanduser()
PARTICIPANT_FILE = Path("PATH_TO_PARTICIPANT_DATA.xlsx").expanduser()
PARTICIPANT_SHEET = 0
OUTPUT_DIR = Path("PATH_TO_STATISTICAL_OUTPUTS").expanduser()

EXPECTED_N_PARTICIPANTS = 69
MAKE_FIGURES = True
EXPORT_OUTPUTS = False
OVERWRITE_OUTPUTS = False
BOOTSTRAP_REPS = 10_000
BOOTSTRAP_SEED = 1

# Mapping: analysis variable -> column header in the participant file.
COLUMN_MAP = {
    "subject": "subject", "age": "age", "diagnostic_group": "diagnostic_group",
    "identity": "identity", "self_direction": "self_direction",
    "empathy": "empathy", "intimacy": "intimacy",
    "negative_affectivity": "negative_affectivity", "detachment": "detachment",
    "dissociality": "dissociality", "disinhibition": "disinhibition", "anankastia": "anankastia",
}
# Mapping: diagnostic labels in the input file -> analysis group.
DIAGNOSTIC_LABEL_MAP = {
    "no_mental_disorder": "no_mental_disorder",
    "personality_disorder": "personality_disorder",
    "schizotypal_disorder": "schizotypal_disorder",
}
DOMAIN_COLUMNS = ["identity", "self_direction", "empathy", "intimacy"]
TRAIT_COLUMNS = ["negative_affectivity", "detachment", "dissociality", "disinhibition", "anankastia"]
BEHAVIOR_COLUMNS = ["rt_veff80", "rt_veff50", "rt_mod", "rt_mean", "err_mean", "err_80inv", "antic_mean", "antic_mod"]
CONDITION_ORDER = ["valid_80", "invalid_80", "valid_50", "invalid_50"]
ERP_ROI = ["F3", "Fz", "F4", "C3", "Cz", "C4"]
ERP_WINDOW = (0.240, 0.320)

if EXPORT_OUTPUTS and not RUN_ANALYSES:
    raise ValueError("Enable RUN_ANALYSES before exporting study results.")
print("Mode:", "study analysis" if RUN_ANALYSES else "no study data loaded")

Mode: no study data loaded


## Participant data and score definitions

| Analysis column | Contents |
|---|---|
| `subject` | Pseudonymous identifier matching notebooks 01 and 02 |
| `age` | Age in years |
| `diagnostic_group` | One of the three groups defined in `DIAGNOSTIC_LABEL_MAP` |
| `identity`, `self_direction` | SIFS self-functioning domain scores |
| `empathy`, `intimacy` | SIFS interpersonal-functioning domain scores |
| Five columns in `TRAIT_COLUMNS` | FFiCD domain scores in their questionnaire units |

Change the values in `COLUMN_MAP` to match the headers of your participant file. Change the keys of `DIAGNOSTIC_LABEL_MAP` to match its diagnostic labels. Unmapped diagnoses stop processing.

All standardization is performed across participants, using the sample standard deviation (`ddof=1`). Age is standardized across the EEG cohort. Each SIFS domain is standardized separately; the two domain z-scores are averaged to form each functioning composite, which is then standardized again for the models. Domain-pair means use the available value when only one domain is observed; their coverage is reported. FFiCD scores are standardized separately across participants with available values.

Table 1 uses the raw means of the two SIFS domains. The `Participant_scores` output sheet preserves the model scores for subsequent CNV analyses; those scores should not be restandardized on a smaller CNV sample.

In [3]:
def normalize_ids(table):
    table = table.copy()
    if "subject" not in table or table["subject"].isna().any():
        raise ValueError("Participant identifiers are required.")
    table["subject"] = table["subject"].astype(str).str.strip().str.upper()
    if table["subject"].eq("").any():
        raise ValueError("Participant identifiers must not be blank.")
    return table


def load_statistical_inputs():
    erp = normalize_ids(pd.read_excel(EEG_FEATURES_FILE, sheet_name="ERP_trials", engine="openpyxl"))
    required_erp = {"trial", "condition", "probability", "validity", "erp_amplitude_uv"}
    if missing := required_erp - set(erp):
        raise ValueError(f"Missing EEG fields: {sorted(missing)}")
    subjects = sorted(erp["subject"].unique())
    if len(subjects) != EXPECTED_N_PARTICIPANTS:
        raise ValueError("The EEG cohort size differs from EXPECTED_N_PARTICIPANTS.")
    if erp.duplicated(["subject", "trial"]).any():
        raise ValueError("Duplicate participant/trial records in ERP_trials.")
    for name in ["probability", "erp_amplitude_uv"]:
        erp[name] = pd.to_numeric(erp[name], errors="raise")
        if not np.isfinite(erp[name]).all(): raise ValueError(f"Invalid EEG values in {name}.")
    if not erp["probability"].isin([50, 80]).all() or not erp["validity"].isin(["valid", "invalid"]).all():
        raise ValueError("Unknown EEG probability or validity level.")
    expected_condition = erp["validity"] + "_" + erp["probability"].astype(int).astype(str)
    if not erp["condition"].eq(expected_condition).all():
        raise ValueError("EEG condition labels disagree with probability/validity.")
    condition_counts = erp.groupby("subject")["condition"].nunique()
    if not condition_counts.eq(4).all():
        incomplete = condition_counts.index[condition_counts.ne(4)].tolist()
        raise ValueError(f"Four ERP conditions are required for every participant: {incomplete}")

    sheets = ["Behavioral_features", "Subject_contrasts", "Participant_condition", "Participant_probability", "Table2_numeric", "Table2_formatted"]
    behavior = pd.read_excel(BEHAVIOR_SUMMARY_FILE, sheet_name=sheets, engine="openpyxl")
    for name in sheets[:4]:
        behavior[name] = normalize_ids(behavior[name])
        if set(behavior[name]["subject"]) != set(subjects):
            raise ValueError(f"Participant mismatch between EEG and {name}.")
    if set(BEHAVIOR_COLUMNS) - set(behavior["Behavioral_features"]):
        raise ValueError("The behavioral predictor table is incomplete.")
    for name in ["Behavioral_features", "Subject_contrasts"]:
        if behavior[name]["subject"].duplicated().any():
            raise ValueError(f"Duplicate participant rows in {name}.")
    if behavior["Participant_condition"].duplicated(["subject", "condition"]).any():
        raise ValueError("Duplicate behavioral participant/condition rows.")
    if behavior["Participant_probability"].duplicated(["subject", "probability"]).any():
        raise ValueError("Duplicate behavioral participant/probability rows.")

    raw_participants = pd.read_excel(PARTICIPANT_FILE, sheet_name=PARTICIPANT_SHEET, engine="openpyxl")
    if len(set(COLUMN_MAP.values())) != len(COLUMN_MAP):
        raise ValueError("Each analysis variable must map to a distinct input column.")
    if missing := set(COLUMN_MAP.values()) - set(raw_participants):
        raise ValueError(f"Missing participant columns: {sorted(missing)}. Check COLUMN_MAP.")
    participants = raw_participants[list(COLUMN_MAP.values())].rename(columns={v:k for k,v in COLUMN_MAP.items()})
    participants = normalize_ids(participants)
    if participants["subject"].duplicated().any():
        raise ValueError("Duplicate participant rows in the participant file.")
    if missing := set(subjects) - set(participants["subject"]):
        raise ValueError(f"Missing participant data: {sorted(missing)}")
    extra_participants = participants.loc[~participants["subject"].isin(subjects), ["subject"]].copy()
    participants = participants.set_index("subject").loc[subjects].reset_index()
    labels = participants["diagnostic_group"].astype("string").str.strip()
    participants["diagnostic_group"] = labels.map(DIAGNOSTIC_LABEL_MAP)
    if participants["diagnostic_group"].isna().any():
        unknown = labels.loc[participants["diagnostic_group"].isna()].drop_duplicates().tolist()
        raise ValueError(f"Unmapped diagnostic labels: {unknown}")
    allowed_groups = {"no_mental_disorder", "personality_disorder", "schizotypal_disorder"}
    if not participants["diagnostic_group"].isin(allowed_groups).all():
        raise ValueError("DIAGNOSTIC_LABEL_MAP contains an unsupported analysis group.")
    for column in ["age", *DOMAIN_COLUMNS, *TRAIT_COLUMNS]:
        participants[column] = pd.to_numeric(participants[column], errors="raise")
        if np.isinf(participants[column]).any():
            raise ValueError(f"Infinite participant values in {column}.")
    return erp, behavior, participants, extra_participants

## Descriptive and participant-level calculations

The behavioral analysis uses seven two-sided Wilcoxon signed-rank tests. Rank-biserial effect sizes use the signed rank sums of nonzero differences; medians and complete-pair counts include zero differences. The anticipatory probability contrast is calculated from pooled detected-trial counts within each probability context.

Table 1 compares three diagnostic groups using Kruskal–Wallis tests, with Benjamini–Hochberg correction across eight measures. Behavioral correlations are reported with correction within each set of eight behavioral measures and across all 56 behavior–personality pairs. Undefined tests are identified in the output.

Participant ERP modulation is `(invalid_80 − valid_80) − (invalid_50 − valid_50)`. The self/interpersonal comparison uses a common complete sample, a Meng test of the rank correlations, and participant bootstrap intervals (10,000 resamples; seed 1). Partial rank correlation controls interpersonal impairment when relating ERP modulation to self impairment.

In [4]:
def _subject_index(frame):
    """Return a participant table indexed by its unique subject identifier."""
    if frame['subject'].duplicated().any():
        raise ValueError('Participant tables must contain one row per subject.')
    return frame.set_index('subject')


def _finite_fdr(pvalues):
    """BH correction among finite p-values; undefined tests retain missing q-values."""
    values = np.asarray(pvalues, dtype=float)
    result = np.full(values.shape, np.nan)
    valid = np.isfinite(values)
    if valid.any():
        result[valid] = multipletests(values[valid], method='fdr_bh')[1]
    return result


def prepare_scores(participants):
    """Return (scored participants, standardization QC); use participant SD, ddof=1."""
    out = participants.copy()
    _subject_index(out)
    qc = []

    def standardize(column, destination):
        values = pd.to_numeric(out[column], errors='raise')
        n, mean, sd = values.count(), values.mean(), values.std(ddof=1)
        estimable = n >= 2 and np.isfinite(sd) and sd > 0
        out[destination] = (values - mean) / sd if estimable else np.nan
        qc.append(dict(variable=column, standardized_variable=destination,
                       n_available=n, mean=mean, sd=sd,
                       status='ok' if estimable else 'insufficient_or_constant'))

    for column in ['age'] + list(DOMAIN_COLUMNS) + list(TRAIT_COLUMNS):
        standardize(column, column + '_z')
    pairs = [('self', DOMAIN_COLUMNS[:2]), ('interpersonal', DOMAIN_COLUMNS[2:])]
    for name, domains in pairs:
        out[name + '_raw'] = out[domains].mean(axis=1, skipna=True)
        out[name + '_n_domains'] = out[domains].notna().sum(axis=1)
        out[name + '_comp'] = out[[c + '_z' for c in domains]].mean(axis=1, skipna=True)
        standardize(name + '_comp', name + '_z')
    return out, pd.DataFrame(qc)


def table1(participants):
    """Return numeric summaries, formatted summaries, and eight KW/BH test results."""
    included = participants.loc[participants['self_comp'].notna()]
    group_names = ['no_mental_disorder', 'personality_disorder', 'schizotypal_disorder']
    variables = ['age', 'self_raw', 'interpersonal_raw'] + list(TRAIT_COLUMNS)
    groups = {'all_participants': included}
    groups.update({name: included.loc[included['diagnostic_group'].eq(name)] for name in group_names})
    summaries, tests = [], []
    for variable in variables:
        for name, group in groups.items():
            values = group[variable].dropna()
            summaries.append(dict(variable=variable, group=name, n_group=len(group),
                                  n_available=len(values), mean=values.mean(), sd=values.std(ddof=1),
                                  minimum=values.min(), maximum=values.max()))
        arrays = [groups[name][variable].dropna().to_numpy(float) for name in group_names]
        enough = all(len(values) > 1 for values in arrays)
        nonconstant = enough and np.unique(np.concatenate(arrays)).size > 1
        if nonconstant:
            test = stats.kruskal(*arrays)
            statistic, pvalue, status = test.statistic, test.pvalue, 'ok'
        else:
            statistic, pvalue = np.nan, np.nan
            status = 'constant' if enough else 'insufficient_group_data'
        tests.append(dict(variable=variable, H=statistic, p=pvalue, status=status))
    numeric, kw = pd.DataFrame(summaries), pd.DataFrame(tests)
    kw['q_fdr'] = _finite_fdr(kw['p'])
    kw['family_size_planned'] = len(variables)
    kw['family_size_tested'] = np.isfinite(kw['p']).sum()
    formatted = numeric.assign(value=numeric.apply(
        lambda r: f"{r['mean']:.2f} ± {r['sd']:.2f}" if r['n_available'] > 1 else '—', axis=1))
    formatted = formatted.pivot(index='variable', columns='group', values='value').reindex(
        index=variables, columns=list(groups))
    formatted = formatted.join(kw.set_index('variable')[['p', 'q_fdr']]).reset_index()
    return numeric, formatted, kw


def participant_erp_features(erp_trials, participants):
    """Return participant-level condition means, ERP contrasts, and sampling variances."""
    if not erp_trials['subject'].isin(participants['subject']).all():
        raise ValueError('ERP trials include subjects absent from the participant table.')
    grouped = erp_trials.groupby(['subject', 'probability', 'validity'])['erp_amplitude_uv']
    cells = grouped.agg(['mean', 'var', 'count'])
    out = _subject_index(participants).copy()
    for probability in [50, 80]:
        for validity in ['valid', 'invalid']:
            try:
                cell = cells.xs((probability, validity), level=('probability', 'validity'))
            except KeyError:
                cell = pd.DataFrame(columns=['mean', 'var', 'count'])
            label = f'{validity}_{probability}'
            out[f'erp_{label}_uv'] = cell['mean']
            out[f'erp_{label}_variance_uv2'] = cell['var']
            out[f'n_erp_{label}'] = cell['count'].reindex(out.index, fill_value=0).astype(int)
        out[f'validity_effect_{probability}'] = out[f'erp_invalid_{probability}_uv'] - out[f'erp_valid_{probability}_uv']
        out[f'validity_effect_{probability}_se2'] = sum(
            out[f'erp_{validity}_{probability}_variance_uv2'] / out[f'n_erp_{validity}_{probability}'].replace(0, np.nan)
            for validity in ['valid', 'invalid'])
    out['probability_modulation_uv'] = out['validity_effect_80'] - out['validity_effect_50']
    out['mean_validity_effect'] = (out['validity_effect_80'] + out['validity_effect_50']) / 2
    return out.reset_index()


def _signed_rank(x, y):
    paired = pd.DataFrame({'x': x, 'y': y}).replace([np.inf, -np.inf], np.nan).dropna()
    differences = (paired['x'] - paired['y']).to_numpy(float)
    nonzero = differences[differences != 0]
    result = dict(n_complete=len(differences), n_nonzero=len(nonzero),
                  median_difference=np.median(differences) if len(differences) else np.nan,
                  W=np.nan, p=np.nan, rank_biserial=np.nan, status='insufficient_data')
    if len(differences) and not len(nonzero):
        result.update(W=0.0, p=1.0, rank_biserial=0.0, status='all_differences_zero')
    elif len(nonzero) >= 3:
        ranks = stats.rankdata(np.abs(nonzero))
        positive, negative = ranks[nonzero > 0].sum(), ranks[nonzero < 0].sum()
        test = stats.wilcoxon(paired['x'], paired['y'], alternative='two-sided',
                              zero_method='wilcox', correction=False, method='auto')
        result.update(W=test.statistic, p=test.pvalue,
                      rank_biserial=(positive - negative) / (positive + negative), status='ok')
    return result


def behavior_tests(features, contrasts, participant_condition, participant_probability, subjects):
    """Return seven paired tests; verify pooled anticipation against exported contrasts."""
    ids = pd.Index(subjects, name='subject')
    feature, contrast = _subject_index(features).reindex(ids), _subject_index(contrasts).reindex(ids)
    condition = participant_condition.loc[participant_condition['subject'].isin(ids)]
    probability = participant_probability.loc[participant_probability['subject'].isin(ids)].copy()
    probability['pooled_pct'] = 100 * probability['n_anticipatory_lt90'] / probability['n_detected'].replace(0, np.nan)
    pooled = probability.pivot(index='subject', columns='probability', values='pooled_pct').reindex(index=ids, columns=[50, 80])
    latency = condition.pivot(index='subject', columns='condition', values='mean_srt_correct_ms').reindex(ids)
    errors = condition.pivot(index='subject', columns='condition', values='error_pct_detected').reindex(ids)
    rows = []
    checks = {}
    for prefix, wide, unit in [('rt', latency, 'ms'), ('error', errors, 'percentage_points')]:
        for context in [80, 50]:
            x, y = wide[f'invalid_{context}'], wide[f'valid_{context}']
            key = f'rt_veff{context}' if prefix == 'rt' else f'error_veff{context}_pp'
            checks[key] = x - y
            rows.append(dict(contrast=key, unit=unit, **_signed_rank(x, y)))
        difference = checks[f'rt_veff80' if prefix == 'rt' else 'error_veff80_pp'] - checks[
            f'rt_veff50' if prefix == 'rt' else 'error_veff50_pp']
        key = 'rt_mod' if prefix == 'rt' else 'error_mod_pp'
        checks[key] = difference
        rows.append(dict(contrast=key, unit=unit, **_signed_rank(difference, pd.Series(0.0, index=ids))))
    checks['antic_mod'] = pooled[80] - pooled[50]
    rows.append(dict(contrast='antic_mod', unit='percentage_points', **_signed_rank(pooled[80], pooled[50])))
    for name, values in checks.items():
        if not np.allclose(values, contrast[name], rtol=1e-10, atol=1e-10, equal_nan=True):
            raise ValueError(f'Subject_contrasts disagrees with condition/count data: {name}')
        if name in feature and not np.allclose(values, feature[name], rtol=1e-10, atol=1e-10, equal_nan=True):
            raise ValueError(f'Behavioral_features disagrees with condition/count data: {name}')
    return pd.DataFrame(rows)


def _spearman_pair(x, y):
    paired = pd.DataFrame({'x': x, 'y': y}).replace([np.inf, -np.inf], np.nan).dropna()
    valid = len(paired) >= 3 and paired['x'].nunique() > 1 and paired['y'].nunique() > 1
    if valid:
        result = stats.spearmanr(paired['x'], paired['y'])
        return dict(n=len(paired), rho=result.statistic, p=result.pvalue, status='ok')
    return dict(n=len(paired), rho=np.nan, p=np.nan, status='insufficient_or_constant')


def behavior_correlations(features, participants):
    """Return 8 × 7 correlations, BH within moderator and across all finite tests."""
    moderators = ['self_z', 'interpersonal_z'] + [column + '_z' for column in TRAIT_COLUMNS]
    merged = _subject_index(participants)[moderators].join(_subject_index(features)[list(BEHAVIOR_COLUMNS)], how='left')
    rows = [dict(moderator=moderator, measure=measure,
                 **_spearman_pair(merged[moderator], merged[measure]))
            for moderator in moderators for measure in BEHAVIOR_COLUMNS]
    out = pd.DataFrame(rows)
    out['q_within_moderator'] = out.groupby('moderator', sort=False)['p'].transform(_finite_fdr)
    out['q_all'] = _finite_fdr(out['p'])
    out['family_size_within_planned'] = len(BEHAVIOR_COLUMNS)
    out['family_size_within_tested'] = out.groupby('moderator')['p'].transform(lambda values: np.isfinite(values).sum())
    out['family_size_all_planned'] = len(rows)
    out['family_size_all_tested'] = np.isfinite(out['p']).sum()
    return out


def specificity(participant_erp):
    """Return pairwise correlations, common-sample Meng, bootstrap summary, partial rank."""
    columns = ['probability_modulation_uv', 'self_comp', 'interpersonal_comp']
    data = participant_erp[columns].replace([np.inf, -np.inf], np.nan)
    correlations = pd.DataFrame([
        dict(predictor=predictor, **_spearman_pair(data[columns[0]], data[predictor]))
        for predictor in columns[1:]])
    common = data.dropna()
    n = len(common)
    comparison = dict(n=n, rho_self=np.nan, rho_interpersonal=np.nan, rho_between_predictors=np.nan,
                      delta_rho=np.nan, meng_z=np.nan, p=np.nan, status='insufficient_or_constant')
    partial = dict(n=n, partial_rho=np.nan, t=np.nan, df=n - 3, p=np.nan, status='insufficient_or_constant')
    bootstrap = []
    if n >= 4 and (common.nunique() > 1).all():
        rank_correlation = common.rank(method='average').corr().to_numpy()
        r12, r13, r23 = rank_correlation[0, 1], rank_correlation[0, 2], rank_correlation[1, 2]
        comparison.update(rho_self=r12, rho_interpersonal=r13, rho_between_predictors=r23, delta_rho=r12 - r13)
        mean_squared_r = (r12 ** 2 + r13 ** 2) / 2
        if abs(r12) < 1 and abs(r13) < 1 and r23 < 1:
            f = min((1 - r23) / (2 * (1 - mean_squared_r)), 1.0)
            h = (1 - f * mean_squared_r) / (1 - mean_squared_r)
            z = (np.arctanh(r12) - np.arctanh(r13)) * np.sqrt((n - 3) / (2 * (1 - r23) * h))
            comparison.update(meng_z=z, p=2 * stats.norm.sf(abs(z)), status='ok')
        denominator = (1 - r13 ** 2) * (1 - r23 ** 2)
        if denominator > 0:
            rho = np.clip((r12 - r13 * r23) / np.sqrt(denominator), -1.0, 1.0)
            t = rho * np.sqrt((n - 3) / (1 - rho ** 2)) if abs(rho) < 1 else np.copysign(np.inf, rho)
            partial.update(partial_rho=rho, t=t, p=2 * stats.t.sf(abs(t), n - 3), status='ok')
        rng = np.random.default_rng(BOOTSTRAP_SEED)
        estimates = np.full((BOOTSTRAP_REPS, 2), np.nan)
        array = common.to_numpy(float)
        for index in range(BOOTSTRAP_REPS):
            sample = array[rng.choice(n, n, replace=True)]
            if np.unique(sample[:, 0]).size > 1:
                if np.unique(sample[:, 1]).size > 1:
                    estimates[index, 0] = stats.spearmanr(sample[:, 0], sample[:, 1]).statistic
                if np.unique(sample[:, 1]).size > 1 and np.unique(sample[:, 2]).size > 1:
                    estimates[index, 1] = estimates[index, 0] - stats.spearmanr(sample[:, 0], sample[:, 2]).statistic
        for index, (name, estimate) in enumerate([('rho_self', r12), ('delta_rho', r12 - r13)]):
            valid = estimates[:, index][np.isfinite(estimates[:, index])]
            ci = np.percentile(valid, [2.5, 97.5]) if len(valid) else [np.nan, np.nan]
            bootstrap.append(dict(quantity=name, estimate=estimate, ci_low=ci[0], ci_high=ci[1],
                                  n=n, n_resamples=BOOTSTRAP_REPS, n_valid_resamples=len(valid), seed=BOOTSTRAP_SEED,
                                  status='ok' if len(valid) == BOOTSTRAP_REPS else 'degenerate_resamples_omitted'))
    else:
        for name in ['rho_self', 'delta_rho']:
            bootstrap.append(dict(quantity=name, estimate=np.nan, ci_low=np.nan, ci_high=np.nan,
                                  n=n, n_resamples=BOOTSTRAP_REPS, n_valid_resamples=0, seed=BOOTSTRAP_SEED,
                                  status='insufficient_or_constant'))
    return correlations, pd.DataFrame([comparison]), pd.DataFrame(bootstrap), pd.DataFrame([partial])


def attrition_tests(participant_erp):
    """Compare participants with and without a self-functioning composite."""
    has_scores = participant_erp['self_comp'].notna()
    rows = []
    for column in ['age', 'probability_modulation_uv']:
        with_scores = participant_erp.loc[has_scores, column].dropna()
        without_scores = participant_erp.loc[~has_scores, column].dropna()
        valid = len(with_scores) > 0 and len(without_scores) > 0
        test = stats.mannwhitneyu(with_scores, without_scores, alternative='two-sided', method='auto') if valid else None
        rows.append(dict(variable=column, n_with_scores=len(with_scores), n_without_scores=len(without_scores),
                         U=test.statistic if valid else np.nan, p=test.pvalue if valid else np.nan,
                         status='ok' if valid else 'empty_group'))
    return pd.DataFrame(rows)

## Trial-level mixed models

Validity is coded −0.5 for valid and +0.5 for invalid trials. Probability is coded −0.5 for 50% and +0.5 for 80%. The outcome is P3a amplitude in microvolts.

| Model | Fixed-effects formula |
|---|---|
| M0 | `amplitude ~ validity_c * probability_c` |
| M1 | `amplitude ~ validity_c * probability_c * self_z + age_z` |
| M2 | `amplitude ~ validity_c * probability_c * interpersonal_z + age_z` |
| FFiCD, five models | `amplitude ~ validity_c * probability_c * trait_z + age_z` |
| Joint, two models | `amplitude ~ validity_c * probability_c * (trait_z + self_z) + age_z` |

Every model includes a participant random intercept and random slopes for validity, probability, and their interaction, with a full covariance matrix. Models use restricted maximum likelihood and the Powell optimizer. The joint models concern negative affectivity and disinhibition. They do not include a trait-by-self interaction.

The five FFiCD three-way interaction p-values form one Benjamini–Hochberg family. Fixed-effect intervals and p-values use the model's normal approximation. Convergence, warnings, retained sample sizes, and the random-effect covariance matrices are exported for inspection.

In [5]:
def fixed_effect_table(model, model_id):
    terms = model.fe_params.index
    intervals = model.conf_int().loc[terms]
    return pd.DataFrame({
        "model": model_id, "term": terms, "b": model.fe_params.to_numpy(),
        "SE": model.bse.loc[terms].to_numpy(), "z": model.tvalues.loc[terms].to_numpy(),
        "p": model.pvalues.loc[terms].to_numpy(),
        "ci_lower": intervals.iloc[:, 0].to_numpy(), "ci_upper": intervals.iloc[:, 1].to_numpy(),
    })


def fit_mixed_model(data, model_id, formula, required_columns):
    complete = data.dropna(subset=required_columns).copy()
    if complete["subject"].nunique() < 3:
        raise ValueError(f"{model_id}: insufficient participants with complete model data.")
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        model = smf.mixedlm(formula, complete, groups=complete["subject"],
                            re_formula="~ validity_c * probability_c").fit(reml=True, method="powell")
    covariance = model.cov_re
    eigenvalues = np.linalg.eigvalsh(covariance.to_numpy())
    interaction = "validity_c:probability_c"
    diag = {
        "model": model_id, "formula": formula, "converged": bool(model.converged),
        "n_subjects": complete["subject"].nunique(), "n_trials": len(complete),
        "excluded_subjects": data["subject"].nunique() - complete["subject"].nunique(),
        "excluded_trials": len(data) - len(complete), "log_likelihood": model.llf,
        "residual_variance_uv2": model.scale, "minimum_random_cov_eigenvalue": eigenvalues.min(),
        "random_probability_modulation_sd_uv": np.sqrt(covariance.loc[interaction, interaction]),
        "warning_count": len(caught),
    }
    warning_rows = [{"model": model_id, "warning": str(w.message), "category": w.category.__name__} for w in caught]
    covariance_rows = [{"model": model_id, "row": row, "column": col, "covariance": covariance.loc[row, col]}
                       for row in covariance.index for col in covariance.columns]
    return model, fixed_effect_table(model, model_id), diag, warning_rows, covariance_rows


def linear_contrast(model, weights):
    vector = pd.Series(0.0, index=model.fe_params.index)
    for term, weight in weights.items(): vector.loc[term] = weight
    covariance = model.cov_params().loc[vector.index, vector.index]
    estimate = float(vector @ model.fe_params)
    variance = float(vector @ covariance @ vector)
    if variance < 0: raise ValueError("Negative contrast variance; inspect the model covariance matrix.")
    se = np.sqrt(variance)
    z = estimate / se if se > 0 else np.nan
    return {"b": estimate, "SE": se, "z": z, "p": 2 * stats.norm.sf(abs(z)),
            "ci_lower": estimate - 1.96 * se, "ci_upper": estimate + 1.96 * se}


def run_erp_models(erp_trials, participants):
    score_columns = ["subject", "age_z", "self_z", "interpersonal_z", *[t + "_z" for t in TRAIT_COLUMNS]]
    data = erp_trials.merge(participants[score_columns], on="subject", how="left", validate="many_to_one")
    data = data.rename(columns={"erp_amplitude_uv": "amplitude"})
    data["validity_c"] = data["validity"].map({"valid": -0.5, "invalid": 0.5})
    data["probability_c"] = data["probability"].map({50: -0.5, 80: 0.5})
    models, fixed, diagnostics, warnings_rows, covariance_rows = {}, [], [], [], []

    def fit(model_id, formula, columns, model_data=data):
        print(f"Fitting {model_id}...")
        result, effect_table, diag, caught, cov_rows = fit_mixed_model(
            model_data, model_id, formula, ["amplitude", "validity_c", "probability_c", *columns])
        models[model_id] = result
        fixed.append(effect_table)
        diagnostics.append(diag)
        warnings_rows.extend(caught)
        covariance_rows.extend(cov_rows)
        return result, effect_table

    fit("M0", "amplitude ~ validity_c * probability_c", [])
    self_model, _ = fit("M1", "amplitude ~ validity_c * probability_c * self_z + age_z", ["self_z", "age_z"])
    fit("M2", "amplitude ~ validity_c * probability_c * interpersonal_z + age_z", ["interpersonal_z", "age_z"])
    trait_rows = []
    for trait in TRAIT_COLUMNS:
        model_data = data.assign(trait_z=data[trait + "_z"])
        _, result = fit("FFiCD_" + trait, "amplitude ~ validity_c * probability_c * trait_z + age_z",
                        ["trait_z", "age_z"], model_data)
        row = result.loc[result["term"].eq("validity_c:probability_c:trait_z")].iloc[0].to_dict()
        row["trait"] = trait
        trait_rows.append(row)
    table4 = pd.DataFrame(trait_rows)
    table4["q_fdr"] = _finite_fdr(table4["p"])
    table4["family_size_planned"] = len(TRAIT_COLUMNS)
    table4["family_size_tested"] = np.isfinite(table4["p"]).sum()

    joint_rows, joint_correlations = [], []
    for trait in ["negative_affectivity", "disinhibition"]:
        model_data = data.assign(trait_z=data[trait + "_z"])
        _, result = fit("Joint_" + trait,
            "amplitude ~ validity_c * probability_c * (trait_z + self_z) + age_z",
            ["trait_z", "self_z", "age_z"], model_data)
        focus = result.loc[result["term"].isin(["validity_c:probability_c:trait_z", "validity_c:probability_c:self_z"])].copy()
        focus["trait"] = trait
        joint_rows.append(focus)
        participant_values = model_data.dropna(subset=["trait_z", "self_z", "age_z"]).drop_duplicates("subject")
        joint_correlations.append({"trait": trait, "n": len(participant_values),
            "pearson_r": participant_values["trait_z"].corr(participant_values["self_z"]),
            "spearman_rho": participant_values["trait_z"].corr(participant_values["self_z"], method="spearman")})
    simple_slopes = pd.DataFrame([
        {"probability": probability, "effect": "self impairment on validity effect",
         **linear_contrast(self_model, {"validity_c:self_z": 1, "validity_c:probability_c:self_z": weight})}
        for probability, weight in [(50, -0.5), (80, 0.5)]
    ])
    all_fixed = pd.concat(fixed, ignore_index=True)
    return models, {
        "Table3_P3a_models": all_fixed.loc[all_fixed["model"].isin(["M0", "M1", "M2"])].copy(),
        "Table4_FFiCD": table4, "Table5_joint_models": pd.concat(joint_rows, ignore_index=True),
        "Self_simple_slopes": simple_slopes, "Joint_trait_correlations": pd.DataFrame(joint_correlations),
        "All_fixed_effects": all_fixed, "Model_QC": pd.DataFrame(diagnostics),
        "Model_warnings": pd.DataFrame(warnings_rows, columns=["model", "warning", "category"]),
        "Random_covariances": pd.DataFrame(covariance_rows),
    }

## Figures 2 and 3

Figure 2 combines participant-weighted condition waveforms, the scalp distribution of probability modulation, participant validity effects, and correlations with self and interpersonal impairment. Waveform ribbons show ±1 SEM across participants. Validity-effect bars show 95% intervals computed as mean ±1.96 × sample SD/√n. Scatterplot axes use the final standardized functioning scores.

Figure 3 is descriptive. Each validity-effect interval is the difference between the invalid and valid means ±1.96 × √(variance_invalid/n_invalid + variance_valid/n_valid). The dashed reference is the lower-tertile participant mean. Its uncertainty is not incorporated into individual intervals, and no multiplicity correction is applied. Filled markers identify intervals excluding that reference, not individual diagnoses.

Tertile boundaries are the 1/3 and 2/3 quantiles. Scores on the lower boundary enter the lower tertile; scores on the upper boundary enter the upper tertile. Actual group sizes are recorded. Participants within the middle and upper groups are ordered by their average validity effect across probability contexts.

In [6]:
def load_erp_averages(evoked_dir, subjects):
    """Read four condition averages per participant, preserving channels and times."""
    subjects = list(subjects)
    if not subjects or len(subjects) != len(set(subjects)):
        raise ValueError("Provide a nonempty list of unique participant identifiers.")
    averages, reference = {}, None
    for subject in subjects:
        path = Path(evoked_dir) / f"{subject}_erp-ave.fif"
        evokeds = mne.read_evokeds(path, baseline=None, verbose="ERROR")
        comments = [e.comment for e in evokeds]
        if len(comments) != 4 or set(comments) != set(CONDITION_ORDER):
            raise ValueError(f"{subject}: expected exactly one average for each of {CONDITION_ORDER}.")
        averages[subject] = {e.comment: e for e in evokeds}
        for evoked in evokeds:
            if not set(ERP_ROI).issubset(evoked.ch_names):
                raise ValueError(f"{subject}: missing P3a ROI channels.")
            if reference is None:
                reference = evoked
            if (evoked.ch_names != reference.ch_names or evoked.data.shape[1] != reference.data.shape[1]
                    or evoked.first != reference.first
                    or evoked.info["sfreq"] != reference.info["sfreq"]):
                raise ValueError("All averages must have identical channel order, sampling frequency, and sample grid.")
    return averages


def _figure_style():
    plt.rcParams.update({"font.family": "sans-serif", "font.size": 9,
                         "axes.labelsize": 10, "xtick.labelsize": 8, "ytick.labelsize": 8,
                         "axes.linewidth": 0.8, "pdf.fonttype": 42, "ps.fonttype": 42})


def _despine(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(direction="out", length=3, width=0.8)


def _panel_letter(ax, text, x=-0.12, y=1.02):
    ax.text(x, y, text, transform=ax.transAxes, fontsize=12, fontweight="bold", va="bottom")


def _scatter_modulation(ax, frame, predictor, xlabel, ylabel=None):
    data = frame[[predictor, "probability_modulation_uv"]].dropna()
    if len(data) < 3 or data[predictor].nunique() < 2 or data.iloc[:, 1].nunique() < 2:
        raise ValueError(f"Insufficient nonconstant paired observations for {predictor}.")
    x, y = data[predictor].to_numpy(), data["probability_modulation_uv"].to_numpy()
    ax.scatter(x, y, s=20, facecolors="none", edgecolors="0.15", linewidths=0.8)
    xs = np.array([x.min(), x.max()])
    ax.plot(xs, np.polyval(np.polyfit(x, y, 1), xs), color="black", lw=1.2)
    rho, p = stats.spearmanr(x, y)
    p_text = "p < .001" if p < .001 else f"p = {p:.3f}"
    ax.text(1, 1.02, f"ρ = {rho:.2f}, {p_text}", transform=ax.transAxes,
            ha="right", va="bottom", fontsize=8)
    ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    _despine(ax)


def figure2(averages, participant_erp):
    """Plot group ERPs, the probability contrast, and participant associations."""
    _figure_style()
    frame = participant_erp.set_index("subject", verify_integrity=True)
    if set(frame.index) != set(averages) or len(frame) < 2:
        raise ValueError("Figure 2 requires matching participant identifiers in both inputs and N >= 2.")
    subjects, traces, differences = list(frame.index), {}, []
    reference = averages[subjects[0]][CONDITION_ORDER[0]]
    # Integer sample indices preserve window boundaries after FIF serialization.
    sample_times = (np.arange(reference.data.shape[1]) + reference.first) / reference.info["sfreq"]
    times = sample_times * 1000
    window = (sample_times >= ERP_WINDOW[0]) & (sample_times <= ERP_WINDOW[1])
    if not window.any():
        raise ValueError("The P3a window contains no samples.")
    for condition in CONDITION_ORDER:
        traces[condition] = np.array([
            averages[s][condition].get_data(picks=ERP_ROI).mean(axis=0) * 1e6 for s in subjects
        ])
        column = f"erp_{condition}_uv"
        expected = frame[column].to_numpy(dtype=float)
        measured = traces[condition][:, window].mean(axis=1)
        # FIF averages are stored at finite precision; compare in microvolts.
        agrees = np.isclose(measured, expected, rtol=1e-5, atol=1e-5)
        if not agrees.all():
            mismatched = frame.index[~agrees].tolist()
            raise ValueError(
                f"{condition}: Evoked ROI means disagree with the ERP trial summary for {mismatched}. "
                "Use averages and trial features from the same preprocessing run and measurement window."
            )
    for subject in subjects:
        conditions = averages[subject]
        difference = mne.combine_evoked(
            [conditions[c] for c in ["invalid_80", "valid_80", "invalid_50", "valid_50"]],
            weights=[1, -1, -1, 1],
        ).pick("eeg")
        locations = np.array([channel["loc"][:3] for channel in difference.info["chs"]])
        if (difference.get_montage() is None or not np.isfinite(locations).all()
                or np.any(np.all(np.isclose(locations, 0), axis=1))):
            raise ValueError(f"{subject}: valid recorded electrode coordinates are required for the topography.")
        differences.append(difference)
    # Bad EEG channels are interpolated using the stored electrode geometry.
    grand = mne.grand_average(differences, interpolate_bads=True, drop_bads=True)
    topography = grand.data[:, window].mean(axis=1) * 1e6
    vmax = max(0.5, np.ceil(np.abs(topography).max() * 2) / 2)
    fig = plt.figure(figsize=(7.09, 5.4))
    grid = fig.add_gridspec(2, 3, height_ratios=[1.15, 1], hspace=.55, wspace=.48,
                            left=.09, right=.985, top=.91, bottom=.12)
    ax_a, ax_b = fig.add_subplot(grid[0, :2]), fig.add_subplot(grid[0, 2])
    ax_c, ax_d = fig.add_subplot(grid[1, 0]), fig.add_subplot(grid[1, 1])
    ax_e = fig.add_subplot(grid[1, 2], sharey=ax_d)
    ax_a.axvspan(*(np.array(ERP_WINDOW) * 1000), color=".94", zorder=0)
    styles = {"valid_80": ("black", "-"), "invalid_80": ("black", "--"),
              "valid_50": (".55", "-"), "invalid_50": (".55", "--")}
    for condition in CONDITION_ORDER:
        values = traces[condition]
        mean, sem = values.mean(axis=0), values.std(axis=0, ddof=0) / np.sqrt(len(subjects))
        ax_a.fill_between(times, mean - sem, mean + sem, color=".87", linewidth=0, zorder=1)
    for condition in CONDITION_ORDER:
        validity, probability = condition.split("_")
        color, line = styles[condition]
        ax_a.plot(times, traces[condition].mean(axis=0), color=color, ls=line, lw=1.3,
                  label=f"{probability}% {validity}", zorder=3)
    ax_a.axhline(0, color=".75", lw=.6)
    ax_a.axvline(0, color=".75", lw=.6)
    ax_a.set(xlim=(times[0], times[-1]), xlabel="Time relative to target onset (ms)", ylabel="Amplitude (µV)")
    low, high = ax_a.get_ylim()
    ax_a.set_ylim(low, high + .22 * (high - low))
    ax_a.text(.99, .97, "Fronto-central ROI\n(F3, Fz, F4, C3, Cz, C4)", transform=ax_a.transAxes,
              ha="right", va="top", fontsize=7, color=".35")
    ax_a.legend(loc="upper left", ncol=2, fontsize=7, frameon=False, handlelength=2)
    _despine(ax_a)
    mask = np.array([ch in ERP_ROI for ch in grand.ch_names])
    image, _ = mne.viz.plot_topomap(topography, grand.info, axes=ax_b, show=False,
        cmap="RdBu_r", vlim=(-vmax, vmax), contours=6, sensors="k.", mask=mask,
        mask_params=dict(marker="o", markerfacecolor="white", markeredgecolor="black",
                         markeredgewidth=.5, linewidth=0, markersize=3.5),
        outlines="head", extrapolate="head")
    ax_b.set_title("(Invalid − Valid)\n80% − 50%", fontsize=8, pad=3)
    cax = ax_b.inset_axes([.18, -.10, .64, .045])
    bar = fig.colorbar(image, cax=cax, orientation="horizontal", ticks=[-vmax, 0, vmax])
    bar.set_label("µV (240–320 ms)", fontsize=7, labelpad=1)
    cax.tick_params(labelsize=7, length=2, width=.5)
    dv80, dv50 = frame["validity_effect_80"].to_numpy(), frame["validity_effect_50"].to_numpy()
    if not np.isfinite(np.r_[dv80, dv50]).all():
        raise ValueError("Every participant must have both validity effects for Figure 2.")
    for a, b in zip(dv80, dv50):
        ax_c.plot([0, 1], [a, b], color=".8", lw=.6, zorder=1)
    for position, values, color, marker in [(0, dv80, "black", "o"), (1, dv50, ".4", "s")]:
        ax_c.errorbar(position, values.mean(), yerr=1.96 * values.std(ddof=1) / np.sqrt(len(values)),
                       fmt=marker, color=color, ms=5, capsize=2.5, lw=1.2, zorder=3)
    ax_c.axhline(0, color=".75", lw=.6, ls=":")
    ax_c.set(xlim=(-.35, 1.35), xlabel="Probability", ylabel="Invalid − Valid (µV)")
    ax_c.set_xticks([0, 1], ["80%", "50%"])
    _despine(ax_c)
    _scatter_modulation(ax_d, frame, "self_z", "Self impairment\n(z)", "Probability modulation (µV)")
    _scatter_modulation(ax_e, frame, "interpersonal_z", "Interpersonal impairment\n(z)")
    ax_e.tick_params(labelleft=False)
    for ax, letter, x, y in [(ax_a, "A", -.06, 1.02), (ax_b, "B", -.02, 1.15),
                              (ax_c, "C", -.2, 1.02), (ax_d, "D", -.2, 1.02), (ax_e, "E", -.2, 1.02)]:
        _panel_letter(ax, letter, x, y)
    return fig


def figure3(participant_erp):
    """Plot descriptive individual intervals relative to the lower-tertile mean."""
    _figure_style()
    frame = participant_erp.set_index("subject", verify_integrity=True).dropna(
        subset=["self_z", "validity_effect_80", "validity_effect_50"])
    if len(frame) < 3:
        raise ValueError("Figure 3 requires at least three participants with psychometric and ERP data.")
    lower, upper = frame["self_z"].quantile([1 / 3, 2 / 3])
    if lower >= upper:
        raise ValueError("Self-score quantiles coincide; three distinct tertile groups cannot be defined.")
    reference = frame[frame["self_z"] <= lower]
    middle = frame[(frame["self_z"] > lower) & (frame["self_z"] < upper)].sort_values("mean_validity_effect")
    high = frame[frame["self_z"] >= upper].sort_values("mean_validity_effect")
    if middle.empty or high.empty:
        raise ValueError("Figure 3 requires nonempty middle and upper tertiles.")
    selected = pd.concat([middle.assign(tertile="Middle"), high.assign(tertile="Upper")])
    positions = np.r_[np.arange(len(middle)), np.arange(len(high)) + len(middle) + 1]
    fig, axes = plt.subplots(1, 2, figsize=(7.09, 5), sharex=True, sharey=True)
    fig.subplots_adjust(left=.08, right=.87, top=.92, bottom=.14, wspace=.12)
    records = []
    for ax, probability in zip(axes, [80, 50]):
        effect = selected[f"validity_effect_{probability}"].to_numpy()
        se2 = selected[f"validity_effect_{probability}_se2"].to_numpy()
        if not np.isfinite(se2).all() or (se2 < 0).any():
            raise ValueError("Each plotted effect requires a finite nonnegative trial-based variance estimate.")
        half_width = 1.96 * np.sqrt(se2)
        ref_mean = reference[f"validity_effect_{probability}"].mean()
        excludes = np.abs(effect - ref_mean) > half_width
        ax.axvline(0, color=".75", lw=.6)
        ax.axhline(len(middle), color=".7", lw=.6, ls=":")
        ax.axvline(ref_mean, color="black", lw=.9, ls="--")
        ax.errorbar(effect, positions, xerr=half_width, fmt="none", ecolor=".55", elinewidth=.8)
        ax.scatter(effect, positions, s=18, facecolors=np.where(excludes, "black", "white"),
                   edgecolors=".15", linewidths=.8, zorder=3)
        ax.set(title=f"{probability}% context", xlabel="Invalid − Valid (µV)")
        _despine(ax)
        for order, (subject, row) in enumerate(selected.iterrows()):
            records.append(dict(subject=subject, probability=probability, tertile=row["tertile"],
                order=order, plot_y=int(positions[order]), self_z=row["self_z"],
                tertile_lower_boundary=lower, tertile_upper_boundary=upper,
                validity_effect_uv=effect[order], ci_lower_uv=effect[order] - half_width[order],
                ci_upper_uv=effect[order] + half_width[order], lower_tertile_mean_uv=ref_mean,
                lower_tertile_n=len(reference), ci_excludes_reference=bool(excludes[order])))
    axes[0].set(ylim=(-1, positions.max() + 1), yticks=[], ylabel="Participants")
    for y, text in [((len(middle) - 1) / 2, "Middle\ntertile"),
                     (len(middle) + 1 + (len(high) - 1) / 2, "Upper\ntertile")]:
        axes[1].text(1.03, y, text, transform=axes[1].get_yaxis_transform(), va="center", fontsize=9)
    _panel_letter(axes[0], "A", -.06)
    _panel_letter(axes[1], "B", -.04)
    return fig, pd.DataFrame(records)

In [7]:
result_tables = None
models = {}
figures = {}
if RUN_ANALYSES:
    erp_trials, behavior, participant_input, extra_participants = load_statistical_inputs()
    participants, score_qc = prepare_scores(participant_input)
    participant_erp = participant_erp_features(erp_trials, participants)
    table1_numeric, table1_formatted, table1_tests = table1(participants)
    group_behavior = behavior_tests(behavior["Behavioral_features"], behavior["Subject_contrasts"],
                                    behavior["Participant_condition"], behavior["Participant_probability"],
                                    participants["subject"].tolist())
    behavioral_correlations = behavior_correlations(behavior["Behavioral_features"], participants)
    erp_correlations, comparison, bootstrap, partial = specificity(participant_erp)
    attrition = attrition_tests(participant_erp)
    models, model_tables = run_erp_models(erp_trials, participants)
    result_tables = {
        "Participant_scores": participants, "Participant_ERP": participant_erp, "Score_QC": score_qc,
        "Table1_numeric": table1_numeric, "Table1_formatted": table1_formatted,
        "Table1_tests": table1_tests, "Table2_numeric": behavior["Table2_numeric"],
        "Table2_formatted": behavior["Table2_formatted"], "Behavior_group_tests": group_behavior,
        "Behavior_correlations": behavioral_correlations, "ERP_correlations": erp_correlations,
        "Correlation_comparison": comparison, "Bootstrap_intervals": bootstrap,
        "Partial_rank_correlation": partial, "Psychometric_attrition": attrition,
        "Extra_participants": extra_participants, "Software_versions": software_versions,
        **model_tables,
    }
    if MAKE_FIGURES:
        averages = load_erp_averages(EVOKED_DIR, participants["subject"].tolist())
        figures["Figure2_P3a"] = figure2(averages, participant_erp)
        figures["Figure3_individual_profiles"], profiles = figure3(participant_erp)
        result_tables["Individual_profiles"] = profiles
        plt.show()
    settings = {
        "standardization_ddof": 1, "SIFS_model_score": "domain z-scores -> mean of available pair -> z-score",
        "SIFS_descriptive_score": "mean of available raw domain pair", "age_standardization": "ERP participant cohort",
        "random_effects": "intercept + validity + probability + validity:probability; full covariance",
        "estimation": "REML", "optimizer": "Powell", "bootstrap_replicates": BOOTSTRAP_REPS,
        "bootstrap_seed": BOOTSTRAP_SEED, "FFiCD_FDR_family": "five three-way interactions",
        "behavior_FDR_families": "eight tests per moderator; all 56 tests",
        "Table1_FDR_family": "eight Kruskal-Wallis tests",
    }
    result_tables["Analysis_settings"] = pd.DataFrame([(key, str(value)) for key, value in settings.items()], columns=["setting", "value"])
    input_rows = []
    for input_path in [BEHAVIOR_SUMMARY_FILE, EEG_FEATURES_FILE, PARTICIPANT_FILE]:
        digest = hashlib.sha256(input_path.read_bytes()).hexdigest()
        input_rows.append({"file": input_path.name, "sha256": digest})
    result_tables["Input_files"] = pd.DataFrame(input_rows)
else:
    display(Markdown("Study analysis is disabled. Configure the input files and set `RUN_ANALYSES = True`."))

Study analysis is disabled. Configure the input files and set `RUN_ANALYSES = True`.

## Inspect results

Review the score-coverage table and all model warnings. A returned coefficient table does not by itself establish convergence. Nonconverged or otherwise problematic fits require inspection before interpretation.

The behavioral probability contrast is in percentage points, whereas latency contrasts are in milliseconds. P3a coefficients are in microvolts; moderator effects are per participant-level standard deviation.

In [8]:
if result_tables is not None:
    display(result_tables["Score_QC"])
    display(result_tables["Model_QC"])
    if not result_tables["Model_warnings"].empty:
        display(result_tables["Model_warnings"])
    for sheet in ["Table1_formatted", "Behavior_group_tests", "Table3_P3a_models", "Table4_FFiCD",
                  "Table5_joint_models", "Self_simple_slopes", "Correlation_comparison", "Partial_rank_correlation"]:
        display(Markdown(f"### {sheet.replace('_', ' ')}"))
        display(result_tables[sheet])
    display(Markdown("### Behavioral correlations with self impairment"))
    display(result_tables["Behavior_correlations"].loc[result_tables["Behavior_correlations"]["moderator"].eq("self_z")])

## Export

`erp_behavior_statistics.xlsx` contains the numerical result tables, participant scores, model diagnostics, input hashes, and analysis settings. The `Participant_scores` sheet is the participant-level input for the CNV statistical notebook.

When figures have been generated, they are exported as vector PDF and 600-dpi TIFF. Result files contain participant-level research information and should be stored separately from the public code. Existing outputs are replaced only when `OVERWRITE_OUTPUTS = True`.

In [9]:
def export_statistics(tables, figures, output_dir, overwrite=False):
    output_dir = Path(output_dir).expanduser()
    if "PATH_TO_" in str(output_dir):
        raise ValueError("Set OUTPUT_DIR before exporting.")
    workbook = output_dir / "erp_behavior_statistics.xlsx"
    targets = [workbook] + [output_dir / f"{name}{extension}" for name in figures for extension in (".pdf", ".tiff")]
    existing = [path.name for path in targets if path.exists()]
    if existing and not overwrite:
        raise FileExistsError(f"Outputs already exist: {existing}. Set OVERWRITE_OUTPUTS to replace them.")
    output_dir.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(workbook, engine="openpyxl") as writer:
        for sheet, table in tables.items():
            table.to_excel(writer, sheet_name=sheet, index=False)
    for name, figure in figures.items():
        figure.savefig(output_dir / f"{name}.pdf", bbox_inches="tight", facecolor="white")
        figure.savefig(output_dir / f"{name}.tiff", dpi=600, bbox_inches="tight", facecolor="white",
                       pil_kwargs={"compression": "tiff_lzw"})
    return pd.DataFrame({"file": [path.name for path in targets]})

if EXPORT_OUTPUTS:
    if result_tables is None:
        raise RuntimeError("Run the statistical analysis before exporting.")
    display(export_statistics(result_tables, figures, OUTPUT_DIR, OVERWRITE_OUTPUTS))
else:
    print("Export disabled; no participant results written.")

Export disabled; no participant results written.
